In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data/raw") if Path("data/raw").exists() else Path("../data/raw")

freq = pd.read_csv(DATA_DIR / "freMTPLfreq.csv")
sev = pd.read_csv(DATA_DIR / "freMTPLsev.csv")

sev_agg = (
    sev.groupby("PolicyID", as_index=False)
       .agg(
           ClaimAmount=("ClaimAmount", "sum"),
           ClaimCountFromSev=("ClaimAmount", "size")
       )
)

df = freq.merge(sev_agg, on="PolicyID", how="left")
df["ClaimAmount"] = df["ClaimAmount"].fillna(0.0)
df["ClaimCountFromSev"] = df["ClaimCountFromSev"].fillna(0).astype(int)

In [ ]:
# mild outlier / data-quality treatment
df["ClaimNb"] = df["ClaimNb"].clip(upper=4)
df["Exposure"] = df["Exposure"].clip(lower=0, upper=1)
df["ClaimAmount"] = df["ClaimAmount"].clip(upper=200_000)

# if total claim amount is zero, force count to zero
df.loc[(df["ClaimAmount"] == 0) & (df["ClaimNb"] > 0), "ClaimNb"] = 0

# drop impossible exposure rows
df = df[df["Exposure"] > 0].copy()

# small feature engineering
df["CarAgeCapped"] = df["CarAge"].clip(upper=20)
df["DriverAgeCapped"] = df["DriverAge"].clip(lower=18, upper=90)
df["LogDensity"] = np.log1p(df["Density"])

# modeling targets
df["Frequency"] = df["ClaimNb"] / df["Exposure"]
df["AvgClaimAmount"] = df["ClaimAmount"] / np.maximum(df["ClaimNb"], 1)
df["PurePremium"] = df["ClaimAmount"] / df["Exposure"]
df["HasClaim"] = (df["ClaimAmount"] > 0).astype(int)

In [ ]:
print(df.shape)
print(df[["ClaimNb", "ClaimCountFromSev", "ClaimAmount", "Exposure"]].describe())
print("Policies with claims:", df["HasClaim"].sum())
print("Mismatch rate ClaimNb vs matched claim rows:",
      (df["ClaimNb"] != df["ClaimCountFromSev"]).mean())

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["HasClaim"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["HasClaim"]
)

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = [
    "CarAgeCapped",
    "DriverAgeCapped",
    "LogDensity",
]

categorical_cols = [
    "Power",
    "Brand",
    "Gas",
    "Region",
]

feature_cols = numeric_cols + categorical_cols

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ],
    remainder="drop",
)

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import PoissonRegressor

def weighted_mae(y_true, y_pred, weights):
    return np.average(np.abs(y_true - y_pred), weights=weights)

best_freq_model = None
best_freq_alpha = None
best_freq_score = np.inf

for alpha in [1e-6, 1e-4, 1e-2]:
    freq_model = Pipeline(steps=[
        ("prep", preprocessor),
        ("model", PoissonRegressor(alpha=alpha, solver="newton-cholesky", max_iter=300))
    ])

    freq_model.fit(
        train_df[feature_cols],
        train_df["Frequency"],
        model__sample_weight=train_df["Exposure"]
    )

    pred_valid_freq = freq_model.predict(valid_df[feature_cols])
    score = weighted_mae(valid_df["Frequency"], pred_valid_freq, valid_df["Exposure"])

    if score < best_freq_score:
        best_freq_score = score
        best_freq_alpha = alpha
        best_freq_model = freq_model

print(best_freq_alpha, best_freq_score)

0.0001 0.13235542721442353


In [23]:
from sklearn.linear_model import GammaRegressor

sev_train = train_df[train_df["ClaimAmount"] > 0].copy()
sev_valid = valid_df[valid_df["ClaimAmount"] > 0].copy()

best_sev_model = None
best_sev_alpha = None
best_sev_score = np.inf

for alpha in [0.1, 1.0, 10.0]:
    sev_model = Pipeline(steps=[
        ("prep", preprocessor),
        ("model", GammaRegressor(alpha=alpha, solver="newton-cholesky", max_iter=300))
    ])

    sev_model.fit(
        sev_train[feature_cols],
        sev_train["AvgClaimAmount"],
        model__sample_weight=sev_train["ClaimNb"]
    )

    pred_valid_sev = sev_model.predict(sev_valid[feature_cols])
    score = weighted_mae(sev_valid["AvgClaimAmount"], pred_valid_sev, sev_valid["ClaimNb"])

    if score < best_sev_score:
        best_sev_score = score
        best_sev_alpha = alpha
        best_sev_model = sev_model

print(best_sev_alpha, best_sev_score)

0.1 1946.4262639611047


In [24]:
def add_predictions(data):
    out = data.copy()
    out["PredFreq"] = best_freq_model.predict(out[feature_cols])
    out["PredSev"] = best_sev_model.predict(out[feature_cols])
    out["PredPurePremiumRate"] = out["PredFreq"] * out["PredSev"]
    out["ObservedPurePremiumRate"] = out["ClaimAmount"] / out["Exposure"]
    return out

train_pred = add_predictions(train_df)
test_pred = add_predictions(test_df)

In [25]:
train_pred["RiskClass"] = (
    pd.qcut(train_pred["PredPurePremiumRate"], q=10, labels=False, duplicates="drop") + 1
)

cut_points = (
    train_pred.groupby("RiskClass")["PredPurePremiumRate"]
              .max()
              .sort_index()
              .tolist()
)

bins = [-np.inf] + cut_points
labels = list(range(1, len(bins)))

test_pred["RiskClass"] = pd.cut(
    test_pred["PredPurePremiumRate"],
    bins=bins,
    labels=labels,
    include_lowest=True
).astype(int)